# 04_Trainv2 — Data Augmentation cho nhận diện cử chỉ tay

Notebook này **không sửa `02_Train.ipynb`**. Giữ nguyên dữ liệu, cách chia train/val/test, kiến trúc TCN/GRU/Transformer và tiêu chí đánh giá. Augmentation chỉ chạy trên batch train, không tạo file dữ liệu mới; checkpoint phiên bản mới nằm trong `Data_321/checkpoints_v2/<AUGMENT_MODE>/`.

**Cách thử:** chạy lần lượt các cell. Mặc định `AUGMENT_MODE = "all"`. Để so sánh từng phương pháp, đổi thành `"rotation"`, `"noise"`, `"mirror"`, `"rotation_noise"`, `"rotation_mirror"` hoặc `"noise_mirror"`, rồi chạy lại từ cell train. Mỗi chế độ có thư mục checkpoint riêng. Đặt `FORCE_RETRAIN = True` nếu muốn train lại chế độ đang chọn.

In [1]:
#cell 1: đọc manifest và dựng tensor giống 02_Train.ipynb (bắt buộc train bằng CUDA)
import numpy as np, pandas as pd, torch
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
m = pd.read_csv(BASE / "manifest.csv")

def feat(f):
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]                                           # lấy cổ tay làm gốc
    scale = np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    lm = lm / scale                                               # chuẩn hóa theo cỡ bàn tay
    pos = lm.reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])
    has = ~np.isnan(pos[:, 0])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

Tmax = int(m.khung.max())
def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32)
    M = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = feat(f)
        X[i, :len(x)] = x
        M[i, :len(x)] = h
    return (torch.tensor(X, device=dev),
            torch.tensor(df.nhan.values, device=dev),
            torch.tensor(M, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}
print("CUDA:", torch.cuda.get_device_name(0), "| Tmax:", Tmax)
for t, (X, y, M) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(M.sum().item()))

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (151, 355, 126) | video: 151 | khung có tay: 19108
val (31, 355, 126) | video: 31 | khung có tay: 3782
test (33, 355, 126) | video: 33 | khung có tay: 3930


In [2]:
#cell 2: định nghĩa đúng 3 kiến trúc từ 02_Train.ipynb
import torch.nn as nn, torch.nn.functional as F

class Block(nn.Module):
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c)
        s.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))

class GRU(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c); s.rnn = nn.GRU(c, c, batch_first=True); s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.rnn(F.relu(s.inp(x)))[0])

class Trans(nn.Module):
    def __init__(s, c=64, T=Tmax):
        super().__init__(); s.inp = nn.Linear(126, c); s.pos = nn.Parameter(torch.randn(1, T, c) * 0.02)
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(c, 4, 128, 0.1, batch_first=True), 2)
        s.out = nn.Linear(c, 3)
    def forward(s, x):
        t = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(t, device=x.device)
        return s.out(s.enc(s.inp(x) + s.pos[:, :t], mask=mask))

MODELS = [("TCN", TCN), ("GRU", GRU), ("Transformer", Trans)]
print("Đã định nghĩa TCN, GRU và Transformer.")

Đã định nghĩa TCN, GRU và Transformer.


In [3]:
#cell 3: augmentation 3D trực tiếp trên batch train, sau đó tính lại vận tốc
import math, time

AUGMENT_MODE = "all"  # none | rotation | noise | mirror | rotation_noise | rotation_mirror | noise_mirror | all
FORCE_RETRAIN = False
VALID_MODES = {"none", "rotation", "noise", "mirror", "rotation_noise", "rotation_mirror", "noise_mirror", "all"}
assert AUGMENT_MODE in VALID_MODES, f"AUGMENT_MODE phải thuộc {sorted(VALID_MODES)}"

# Mỗi clip được xoay một góc ngẫu nhiên nhất quán theo thời gian; tọa độ đã lấy cổ tay làm gốc.
def augment_batch(x, hand_mask, mode):
    pos = x[..., :63].reshape(x.shape[0], x.shape[1], 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    parts = set() if mode == "none" else (set(mode.split("_")) if mode != "all" else {"rotation", "noise", "mirror"})

    if "rotation" in parts:
        b = x.shape[0]
        ax = (torch.rand(b, 1, 1, device=x.device) * 2 - 1) * math.radians(10)
        ay = (torch.rand(b, 1, 1, device=x.device) * 2 - 1) * math.radians(25)
        az = (torch.rand(b, 1, 1, device=x.device) * 2 - 1) * math.radians(20)
        xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
        y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
        x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
        pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)

    if "mirror" in parts:
        flip = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < 0.5)
        pos[..., 0] = torch.where(flip.squeeze(-1), -pos[..., 0], pos[..., 0])

    if "noise" in parts:
        pos = pos + torch.randn_like(pos) * 0.01 * valid

    # Không làm thay đổi các khung mất tay/khung đệm.
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    vel = torch.zeros_like(pos)
    if pos.shape[1] > 1:
        vel[:, 1:] = pos[:, 1:] - pos[:, :-1]
        consecutive = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None, None]
        vel[:, 1:] = vel[:, 1:] * consecutive
    return torch.cat([pos.reshape(x.shape[0], x.shape[1], 63),
                      vel.reshape(x.shape[0], x.shape[1], 63)], dim=-1)

print("Chế độ augmentation:", AUGMENT_MODE)
print("Checkpoint mới:", BASE / "checkpoints_v2" / AUGMENT_MODE)

Chế độ augmentation: all
Checkpoint mới: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\all


In [4]:
#cell 4: train trên CUDA, augmentation chỉ áp dụng cho train; early stopping và checkpoint riêng
import torch.nn.functional as F

CK = BASE / "checkpoints_v2" / AUGMENT_MODE
CK.mkdir(parents=True, exist_ok=True)

def loss_acc(net, X, y, M):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * M).sum() / M.sum().clamp(min=1), (correct & M).sum() / M.sum().clamp(min=1)

def train_v2(name, cls, epochs=60, bs=16, patience=10, force=False):
    torch.manual_seed(0)
    net = cls().to(dev)
    path = CK / f"{name}.pt"
    if path.exists() and not force:
        ck = torch.load(path, map_location=dev, weights_only=False)
        net.load_state_dict(ck["w"])
        print(f"{name}: nạp checkpoint có sẵn | epoch {ck['epoch']} | {path}")
        return net.eval(), ck.get("hist", [])

    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, epochs + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); total = 0.0
        for i in range(0, len(Xt), bs):
            ids = perm[i:i + bs]
            xb, yb, mb = Xt[ids], yt[ids], Mt[ids]
            xb = augment_batch(xb, mb, AUGMENT_MODE)
            loss, _ = loss_acc(net, xb, yb, mb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, *data["val"])
        row = {"ep": ep, "train_loss": total / len(Xt), "val_loss": vl.item(), "val_acc": va.item()}
        hist.append(row)
        if row["val_loss"] < best:
            best, wait = row["val_loss"], 0
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best,
                        "hist": hist, "model": name, "augment_mode": AUGMENT_MODE}, path)
        else:
            wait += 1
        if ep % 5 == 0 or wait >= patience or ep == epochs:
            print(f"{name} ep{ep:02d} | train {row['train_loss']:.4f} | val {row['val_loss']:.4f} | val_acc {row['val_acc']:.3f}")
        if wait >= patience:
            print(f"Dừng sớm: val loss không giảm sau {patience} epoch."); break
    ck = torch.load(path, map_location=dev, weights_only=False)
    net.load_state_dict(ck["w"])
    print(f"{name}: tốt nhất epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f} | {path} | {time.time()-t0:.0f}s")
    return net.eval(), hist

nets_v2, hists_v2 = {}, {}
for name, cls in MODELS:
    nets_v2[name], hists_v2[name] = train_v2(name, cls, force=FORCE_RETRAIN)
    print("-" * 70)
print("Hoàn tất chế độ:", AUGMENT_MODE)

TCN ep05 | train 0.5826 | val 0.4430 | val_acc 0.873
TCN ep10 | train 0.3979 | val 0.3110 | val_acc 0.909
TCN ep15 | train 0.2943 | val 0.3513 | val_acc 0.857
TCN ep20 | train 0.2595 | val 0.2646 | val_acc 0.918
TCN ep25 | train 0.3662 | val 0.3753 | val_acc 0.872
TCN ep30 | train 0.2696 | val 0.2630 | val_acc 0.902
TCN ep35 | train 0.2314 | val 0.3309 | val_acc 0.869
TCN ep40 | train 0.2540 | val 0.3548 | val_acc 0.879
TCN ep45 | train 0.2383 | val 0.2189 | val_acc 0.915
TCN ep50 | train 0.1906 | val 0.2742 | val_acc 0.877
TCN ep55 | train 0.3985 | val 0.3753 | val_acc 0.888
Dừng sớm: val loss không giảm sau 10 epoch.
TCN: tốt nhất epoch 45 | val_loss 0.2189 | C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\all\TCN.pt | 4s
----------------------------------------------------------------------
GRU ep05 | train 0.9723 | val 0.9091 | val_acc 0.662
GRU ep10 | train 0.3827 | val 0.4117 | val_acc 0.851
GRU ep15 | train 0.3905 | val 0.4609 | val_acc 0.792
GRU ep20 | train 0.285

In [5]:
#cell 5: đánh giá cùng tiêu chí cũ — tỉ lệ video có ít nhất một khung đúng ở mỗi ngưỡng
THR = [0.5, 0.6, 0.7, 0.8, 0.9]
X, y, M = data["test"]
all_nets = {f"v2-{k}": v for k, v in nets_v2.items()}

# Nạp checkpoint baseline nếu có để so sánh trực tiếp với 02_Train.ipynb.
for name, cls in MODELS:
    old_path = BASE / "checkpoints" / f"{name}.pt"
    if old_path.exists():
        old_net = cls().to(dev)
        old_ck = torch.load(old_path, map_location=dev, weights_only=False)
        old_net.load_state_dict(old_ck["w"]); old_net.eval()
        all_nets[f"base-{name}"] = old_net
    else:
        print("Không thấy checkpoint baseline:", old_path)

rows = []
with torch.no_grad():
    for name, net in all_nets.items():
        conf, pred = net(X).softmax(-1).max(-1)
        for t in THR:
            correct_confident = (pred == y[:, None]) & (conf >= t) & M
            rows.append({"model": name, "ngưỡng": t,
                         "video có khung đúng (%)": correct_confident.any(1).float().mean().item() * 100})
result = pd.DataFrame(rows)
pivot = result.pivot(index="model", columns="ngưỡng", values="video có khung đúng (%)")
pivot["TB 5 ngưỡng"] = pivot[THR].mean(axis=1)
print("Tỉ lệ video có ít nhất một khung đúng và đủ tin cậy (%), test:")
display(pivot.round(1))

Tỉ lệ video có ít nhất một khung đúng và đủ tin cậy (%), test:


ngưỡng,0.5,0.6,0.7,0.8,0.9,TB 5 ngưỡng
model,,,,,,
base-GRU,100.0,100.0,100.0,100.0,84.8,97.0
base-TCN,100.0,100.0,97.0,93.9,93.9,97.0
base-Transformer,97.0,97.0,97.0,97.0,97.0,97.0
v2-GRU,100.0,100.0,97.0,97.0,81.8,95.2
v2-TCN,100.0,100.0,97.0,93.9,93.9,97.0
v2-Transformer,100.0,100.0,100.0,100.0,90.9,98.2


In [6]:
#cell 6: so sánh độ chính xác và độ phủ khi model tự tin, cho cả clip và nửa đầu clip
L = torch.tensor(meta["test"].khung.values, device=dev)[:, None]
first_half = torch.arange(X.shape[1], device=dev)[None, :] < L / 2
regions = {"cả clip": M, "nửa đầu": M & first_half}
rows = []
with torch.no_grad():
    for name, net in all_nets.items():
        conf, pred = net(X).softmax(-1).max(-1)
        correct = pred == y[:, None]
        for region_name, base_mask in regions.items():
            for t in THR:
                selected = (conf >= t) & base_mask
                n = selected.sum().item()
                acc = ((correct & selected).sum().item() / n * 100) if n else float("nan")
                coverage = n / max(base_mask.sum().item(), 1) * 100
                rows.append({"model": name, "vùng": region_name, "ngưỡng": t,
                             "độ chính xác (%)": acc, "độ phủ (%)": coverage})
metrics = pd.DataFrame(rows)
print("Độ chính xác khi đạt ngưỡng:")
display(metrics.pivot_table(index=["vùng", "model"], columns="ngưỡng", values="độ chính xác (%)").round(1))
print("Độ phủ — tỉ lệ khung có tay đạt ngưỡng:")
display(metrics.pivot_table(index=["vùng", "model"], columns="ngưỡng", values="độ phủ (%)").round(1))

Độ chính xác khi đạt ngưỡng:


ngưỡng                     0.5   0.6   0.7   0.8    0.9
vùng    model                                          
cả clip base-GRU          91.7  92.4  93.0  93.6   94.5
        base-TCN          93.3  94.9  95.6  96.1   98.1
        base-Transformer  85.7  88.9  92.6  97.2   98.9
        v2-GRU            92.7  93.2  95.0  95.4   98.1
        v2-TCN            96.0  98.1  98.9  99.4   99.9
        v2-Transformer    96.9  98.5  99.4  99.7   99.8
nửa đầu base-GRU          84.9  86.1  86.9  87.7   88.9
        base-TCN          88.1  90.6  91.8  92.4   95.8
        base-Transformer  78.1  83.5  89.4  95.4   98.5
        v2-GRU            86.4  87.1  90.3  90.5   95.2
        v2-TCN            93.2  96.9  98.6  99.3   99.9
        v2-Transformer    95.0  98.1  99.3  99.9  100.0

Độ phủ — tỉ lệ khung có tay đạt ngưỡng:


ngưỡng                     0.5   0.6   0.7   0.8   0.9
vùng    model                                         
cả clip base-GRU          96.9  94.5  91.8  88.1  68.7
        base-TCN          94.7  90.3  87.0  81.6  73.7
        base-Transformer  98.4  94.3  89.4  83.9  79.1
        v2-GRU            96.0  93.8  89.0  82.7  61.7
        v2-TCN            92.1  87.5  81.8  77.6  68.4
        v2-Transformer    93.3  89.2  85.4  81.5  68.6
nửa đầu base-GRU          94.1  89.5  85.0  78.9  57.3
        base-TCN          90.1  82.7  77.2  71.5  58.7
        base-Transformer  97.0  89.4  81.9  75.4  67.6
        v2-GRU            92.7  89.3  80.8  73.0  48.5
        v2-TCN            85.0  76.7  69.8  64.5  54.2
        v2-Transformer    87.2  79.7  73.4  67.7  58.1

## Ghi chú
- Đánh giá dùng nguyên tập test, không áp dụng augmentation lên validation/test.
- Chạy từng `AUGMENT_MODE` và ghi lại bảng kết quả để biết augmentation nào thực sự có ích. Không kết luận chỉ dựa trên độ chính xác train.
- Nếu một checkpoint v2 đã tồn tại, notebook sẽ nạp lại để tránh train lại. Đổi `FORCE_RETRAIN = True` khi muốn chạy lại đúng chế độ đó.
- Đường dẫn checkpoint: `Data_321/checkpoints_v2/<AUGMENT_MODE>/`. Các checkpoint cũ trong `Data_321/checkpoints/` không bị ghi đè.